In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/GaussianDistillation-main/GaussianDistillation-main
%ls

Mounted at /content/drive
/content/drive/MyDrive/GaussianDistillation-main/GaussianDistillation-main
conventions.py  distill_gaussian.py  LICENSE  models/    requirements.txt
datasets.py     __init__.py          Logs/    README.md  utils/


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torchvision
import torch.nn as nn
from torch.optim import Adam
from torchvision import datasets, transforms

# Import the student model
from models.resnet import ResNet18 as ResNet18
import conventions
from utils import misc

# Define distillation loss
criterion = misc.DistillationLoss()  # Custom loss for distillation
xe = nn.CrossEntropyLoss(reduction="mean")  # Cross-entropy loss for validation

# Load CIFAR-10 dataset
def load_cifar10(batch_size):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to match CIFAR-10 distribution
    ])
    train_loader = torch.utils.data.DataLoader(
        datasets.CIFAR10(root="./data", train=True, download=True, transform=transform),
        batch_size=batch_size, shuffle=True
    )
    valid_loader = torch.utils.data.DataLoader(
        datasets.CIFAR10(root="./data", train=False, download=True, transform=transform),
        batch_size=batch_size, shuffle=False
    )
    return train_loader, valid_loader

# Distillation using labeled data
def distill_using_data(teacher_nw, student_nw, train_loader, valid_loader, n_epochs, lr, verbose, device):
    print("\nDistillation using original training data..")
    optimizer = Adam(student_nw.parameters(), lr=lr)
    metrics = []

    for epoch in range(1, n_epochs + 1):
        teacher_nw.eval()
        student_nw.train()
        train_loss = 0.0

        # Training loop
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            with torch.no_grad():
                teacher_output = teacher_nw(data)
            student_output = student_nw(data)
            loss = criterion(student_output, teacher_output)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation loop
        student_nw.eval()
        valid_loss = 0.0
        accs = []
        for data, target in valid_loader:
            data, target = data.to(device), target.to(device)
            with torch.no_grad():
                student_output = student_nw(data)
            loss = xe(student_output, target)
            valid_loss += loss.item()
            accs.append(misc.accuracy_metric(student_output.detach(), target))

        metrics.append([train_loss / len(train_loader), valid_loss / len(valid_loader), np.mean(accs)])
        if verbose and epoch % verbose == 0:
            print("Epoch: {} \tTraining Loss: {:.4f} \tValidation Loss: {:.4f} \tValidation Accuracy: {:.4f}".format(
                epoch, *metrics[-1]
            ))

    return [list(i) for i in zip(*metrics)]  # train_loss, valid_loss, valid_acc

# Distillation using Gaussian noise
def distill_using_noise(model_family, teacher_nw, student_nw, valid_loader, n_epochs, len_batch, lr, verbose, device):
    print("\nDistillation using Gaussian noise..")
    optimizer = Adam(student_nw.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10)
    metrics = []

    data_sample = next(iter(valid_loader))[0]  # Get a single batch shape
    teacher_nw.train()
    student_nw.train()

    for epoch in range(1, n_epochs + 1):
        train_loss = 0.0

        # Training loop with Gaussian noise
        for batch in range(len_batch):
            optimizer.zero_grad()
            data = torch.randn_like(data_sample, device=device) * 0.1 + 0.5  # Gaussian noise scaled and shifted
            with torch.no_grad():
                teacher_output = teacher_nw(data)
            student_output = student_nw(data)
            loss = criterion(student_output, teacher_output)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation loop
        valid_loss = 0.0
        accs = []
        for data, target in valid_loader:
            data, target = data.to(device), target.to(device)
            with torch.no_grad():
                student_output = student_nw(data)
            loss = xe(student_output, target)
            valid_loss += loss.item()
            accs.append(misc.accuracy_metric(student_output.detach(), target))

        scheduler.step(valid_loss)

        metrics.append([train_loss / len_batch, valid_loss / len(valid_loader), np.mean(accs)])
        if verbose and epoch % verbose == 0:
            print("Epoch: {} \tTraining Loss: {:.4f} \tValidation Loss: {:.4f} \tValidation Accuracy: {:.4f}".format(
                epoch, *metrics[-1]
            ))

    return [list(i) for i in zip(*metrics)]  # train_loss, valid_loss, valid_acc

# Experiment setup
def experiment_distil_gaussian(n_epochs_gaussian, n_epochs_data, lr=1e-3, compare=False, verbose=True):
    print("Starting experiment...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load CIFAR-10 dataset
    train_loader, valid_loader = load_cifar10(batch_size=128)
    len_batch = len(train_loader)

    # Load teacher model
    teacher_nw = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet32", pretrained=True)
    teacher_nw.to(device)

    # Load student model
    student_nw = ResNet18()
    student_nw.to(device)

    # Distill using noise
    print("\nStarting Noise-Based Distillation...")
    metrics = distill_using_noise(
        "ResNet",
        teacher_nw,
        student_nw,
        valid_loader,
        n_epochs_gaussian,
        len_batch,
        lr,
        verbose,
        device,
    )
    plt.plot(range(1, len(metrics[2]) + 1), metrics[2], label="Accuracy Noise")

    # Optionally compare with data-based distillation
    if compare:
        student_nw.apply(misc.weight_reset)
        metrics_data = distill_using_data(
            teacher_nw,
            student_nw,
            train_loader,
            valid_loader,
            n_epochs_data,
            lr,
            verbose,
            device,
        )
        plt.plot(range(1, len(metrics_data[2]) + 1), metrics_data[2], label="Accuracy Data")

    plt.title("Student Training")
    plt.legend()
    plt.savefig("accuracy.png", dpi=400)
    plt.close()

# Run the experiment
experiment_distil_gaussian(
    n_epochs_gaussian=20,
    n_epochs_data=20,
    lr=1e-3,
    compare=True,
    verbose=5,
)


Starting experiment...
Using device: cuda


100%|██████████| 170M/170M [00:12<00:00, 13.4MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


/usr/local/lib/python3.10/dist-packages/torch/hub.py:330: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar10_resnet32-ef93fc4d.pt" to /root/.cache/torch/hub/checkpoints/cifar10_resnet32-ef93fc4d.pt
100%|██████████| 1.85M/1.85M [00:00<00:00, 134MB/s]


Starting Noise-Based Distillation...

Distillation using Gaussian noise..
Epoch: 5 	Training Loss: 2.2277 	Validation Loss: 1.8805 	Validation Accuracy: 0.3458
Epoch: 10 	Training Loss: 2.1267 	Validation Loss: 1.6629 	Validation Accuracy: 0.4301
Epoch: 15 	Training Loss: 2.0356 	Validation Loss: 1.4077 	Validation Accuracy: 0.5294
Epoch: 20 	Training Loss: 1.9621 	Validation Loss: 1.2249 	Validation Accuracy: 0.6009

Distillation using original training data..
Epoch: 5 	Training Loss: 0.5578 	Validation Loss: 8.8093 	Validation Accuracy: 0.1224
Epoch: 10 	Training Loss: 0.5221 	Validation Loss: 9.2859 	Validation Accuracy: 0.1263
Epoch: 15 	Training Loss: 0.5027 	Validation Loss: 9.0891 	Validation Accuracy: 0.1277
Epoch: 20 	Training Loss: 0.4917 	Validation Loss: 9.6714 	Validation Accuracy: 0.1288
